In [ ]:
library(tidyverse)
library(lubridate)
library(janitor)

# ============================================================
# CONFIGURAÇÃO
# ============================================================

ARQ_SIM <- "SIM_nao_tratado.csv"
ARQ_CBO <- "lista.csv"
ARQ_CID <- "CID-10-SUBCATEGORIAS.csv"

# ============================================================
# 1. IMPORTAÇÃO
# ============================================================

sim <- read_csv(
  ARQ_SIM,
  locale = locale(encoding = "Latin1"),
  na = c("", "NA", "NULL")
) %>%
  clean_names()

cbo <- read_csv(
  ARQ_CBO,
  locale = locale(encoding = "Latin1")
) %>%
  clean_names()

municipios <- read_csv(
  ARQ_MUN,
  locale = locale(encoding = "UTF-8")
) %>%
  clean_names()

cid <- read_csv(
  ARQ_CID,
  locale = locale(encoding = "UTF-8")
) %>%
  clean_names()


# ============================================================
# 2. TRATAMENTO DAS VARIÁVEIS
# ============================================================

sim <- sim %>%
  mutate(

    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------

    DATA_OBITO = dmy(dtobito),

    # --------------------------------------------------------
    # SEXO
    # --------------------------------------------------------

    SEXO_DESC = case_when(
      sexo %in% c("M", "1") ~ "Masculino",
      sexo %in% c("F", "2") ~ "Feminino",
      sexo %in% c("I", "0", "9") ~ "Ignorado",
      TRUE ~ NA_character_
    ),

    # --------------------------------------------------------
    # RAÇA/COR
    # --------------------------------------------------------

    RACA_DESC = case_when(
      racacor == "1" ~ "Branca",
      racacor == "2" ~ "Preta",
      racacor == "3" ~ "Amarela",
      racacor == "4" ~ "Parda",
      racacor == "5" ~ "Indígena",
      TRUE ~ "Ignorado"
    ),

    # --------------------------------------------------------
    # ESCOLARIDADE
    # --------------------------------------------------------

    ESC_DESC = case_when(
      esc == "1" ~ "Nenhuma",
      esc == "2" ~ "1 a 3 anos",
      esc == "3" ~ "4 a 7 anos",
      esc == "4" ~ "8 a 11 anos",
      esc == "5" ~ "12 anos ou mais",
      esc == "9" ~ "Ignorado",
      TRUE ~ NA_character_
    ),

    # --------------------------------------------------------
    # LOCAL DE OCORRÊNCIA
    # --------------------------------------------------------

    LOCAL_MORTE = case_when(
      lococor == "1" ~ "Hospital",
      lococor == "2" ~ "Outros estabelecimentos de saúde",
      lococor == "3" ~ "Domicílio",
      lococor == "4" ~ "Via pública",
      lococor == "5" ~ "Outros",
      lococor == "6" ~ "Aldeia indígena",
      lococor == "9" ~ "Ignorado",
      TRUE ~ NA_character_
    ),

    # --------------------------------------------------------
    # IDADE
    # --------------------------------------------------------

    idade_cod = as.character(idade),

    idade_unidade = str_sub(idade_cod, 1, 1),

    idade_valor = suppressWarnings(
      as.numeric(str_sub(idade_cod, 2, 3))
    ),

    IDADE_ANOS = case_when(
      idade_unidade == "1" ~ idade_valor / (365.25 * 24 * 60),
      idade_unidade == "2" ~ idade_valor / (365.25 * 24),
      idade_unidade == "3" ~ idade_valor / 12,
      idade_unidade == "4" ~ idade_valor,
      idade_unidade == "5" ~ 100,
      TRUE ~ NA_real_
    )
  )


# ============================================================
# 3. JOIN MUNICÍPIOS
# ============================================================

sim <- sim %>%
  left_join(
    municipios,
    by = c("codmunres" = "codigo_municipio")
  )


# ============================================================
# 4. JOIN CBO
# ============================================================

sim <- sim %>%
  left_join(
    cbo,
    by = c("ocup" = "codigo_cbo")
  )


# ============================================================
# 5. JOIN CID
# ============================================================

sim <- sim %>%
  left_join(
    cid,
    by = c("causabas" = "codigo_cid")
  )


# ============================================================
# 6. DATASET FINAL
# ============================================================

dados_final <- sim %>%
  transmute(
    ANO = as.integer(ano),

    DATA_DE_OBITO = DATA_OBITO,

    IDADE = IDADE_ANOS,

    SEXO = SEXO_DESC,

    RACA = RACA_DESC,

    ESCOLARIDADE = ESC_DESC,

    MUNICIPIO = nome_municipio,

    LOCAL_MORTE = LOCAL_MORTE,

    OCUP = descricao_cbo,

    BASICA_DA_MORTE = descricao_causa
  )


# ============================================================
# 7. EXPORTAÇÃO
# ============================================================

write_csv(
  dados_final,
  "SIM_tratado.csv",
  na = ""
)